#### Initialize Context & Seed Ledger

In [24]:
import sys
from pathlib import Path
HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path: sys.path.insert(0, str(PARENT))

from server.scripts.h3.h3_load import load_geodf_from_csv
PROGRESS_LEDGER = Path("../../map/progress_ledger.csv")
if not PROGRESS_LEDGER.parent.exists():
    raise FileNotFoundError(f"PROGRESS_LEDGER Not Found.")
else:
    print("Loading existing progress ledger")
    progress_geodf = load_geodf_from_csv(PROGRESS_LEDGER)
    print(f"Total Seed Cells: {len(progress_geodf)}")

Loading existing progress ledger
Total Seed Cells: 2959


#### Select Cells

In [42]:
SELECTED_CELLS = []
COORDINATES = [51.524467, -0.097970, 15000]
ID_FIELD = "tile_id"
from server.scripts.h3.h3_selecttiles import select_tiles_in_radius
from server.scripts.h3.h3_visualizemap import visualize_progress

if not SELECTED_CELLS:
    seed_select = select_tiles_in_radius(progress_geodf, COORDINATES[0], COORDINATES[1], COORDINATES[2])
    SELECTED_CELLS = seed_select[ID_FIELD].tolist()

existing_cells = progress_geodf[(progress_geodf['scrubbed']==True)][ID_FIELD].tolist()
SELECTED_CELLS = [CELL for CELL in SELECTED_CELLS if CELL not in existing_cells]

progress_geodf["current"] = False
progress_geodf["current"] = progress_geodf[ID_FIELD].apply(lambda x: x in SELECTED_CELLS)
print(f"Selected {progress_geodf['current'].sum()} cells for Nearby Search.")

Selected 118 cells for Nearby Search.


In [43]:
from server.scripts.h3.h3_load import load_boundary_from_json

print("Updating Ledger")
progress_geodf.to_csv(PROGRESS_LEDGER, index=False)
inner_union = load_boundary_from_json(json_path='../get_seed_map_level1/inner_london_boundary.json')
visualize_progress(progress_geodf, inner_union, output_path=PROGRESS_LEDGER.with_suffix(".html"))

Updating Ledger
Saved Progress Map to: ..\..\map\progress_ledger.html


#### Run APIs

In [44]:
ENABLE_API = True
from server.scripts.get_places.get_places_by_cell import get_places_by_cell
selected_cells = progress_geodf[progress_geodf['current'] == True]
res_geodf = await get_places_by_cell(selected_cells, out_path=Path("../../out/places_cache"))

API calls executed for 333-1: 1 | failures: 0 | places: 10
API calls executed for 337-5: 2 | failures: 0 | places: 12
API calls executed for 339-3: 3 | failures: 0 | places: 14
API calls executed for 339-5: 4 | failures: 0 | places: 22
API calls executed for 339-1: 5 | failures: 0 | places: 22
API calls executed for 347-5: 6 | failures: 0 | places: 29
API calls executed for 347-3: 7 | failures: 0 | places: 31
API calls executed for 347-0: 8 | failures: 0 | places: 32
API calls executed for 357-3: 9 | failures: 0 | places: 44
API calls executed for 357-0-5: 10 | failures: 0 | places: 55
API calls executed for 357-0-3: 11 | failures: 0 | places: 60
API calls executed for 357-4: 12 | failures: 0 | places: 67
API calls executed for 357-0-4: 13 | failures: 0 | places: 73
API calls executed for 357-5: 14 | failures: 0 | places: 76
API calls executed for 357-0-1: 15 | failures: 0 | places: 84
API calls executed for 357-2: 16 | failures: 0 | places: 88
API calls executed for 357-0-0: 17 | fail

In [46]:
cell_geodf = res_geodf.copy()
cell_geodf["tile_id"] = res_geodf["tile_id"].astype(str)
cell_geodf["seed_index"] = res_geodf["seed_index"].astype(int)
cell_geodf["tile_path_id"] = res_geodf["tile_path_id"].astype(str)

#### File API Response and Update Ledger

In [47]:
cell_scrubbed = cell_geodf[cell_geodf['scrubbed'] == True].copy()
if not cell_scrubbed.empty:
    print("Fetch Complete. Updating Ledger.")
    
    # Normalize key type before matching rows between dataframes.
    progress_geodf['tile_id'] = progress_geodf['tile_id'].astype(str)
    cell_scrubbed['tile_id'] = cell_scrubbed['tile_id'].astype(str)

    success_ids = cell_scrubbed['tile_id'].dropna()
    progress_geodf.loc[progress_geodf['tile_id'].isin(success_ids), 'scrubbed'] = True
    
    # Propagate per-tile places_count from this batch into the progress ledger.
    places_count_by_id = (
        cell_scrubbed
        .drop_duplicates(subset=['tile_id'], keep='last')
        .set_index('tile_id')['places_count']
    )
    matched_mask = progress_geodf['tile_id'].isin(places_count_by_id.index)
    progress_geodf.loc[matched_mask, 'places_count'] = progress_geodf.loc[matched_mask, 'tile_id'].map(places_count_by_id)
    
    progress_geodf['current'] = False
    progress_geodf.to_csv(PROGRESS_LEDGER, index=False)
    visualize_progress(progress_geodf, inner_union, output_path=PROGRESS_LEDGER.with_suffix(".html"))
else:
    print("Response InValid")

Fetch Complete. Updating Ledger.
Saved Progress Map to: ..\..\map\progress_ledger.html
